# **Inception-ResNet V1 for Facial Recognition**

**Project Goal**: Using the Inception-ResNet V1 model for facial recognition. Specifically, building a model to recognize the face of Mary Kom.

**Objectives**: 

- Detect faces using MTCNN.
- Create face embeddings with inception-ResNet V1.
- Use created embeddings to recognize faces in images.

In [22]:
import sys 
from pathlib import Path

import matplotlib.pyplot as plt
import PIL 
import torch
import torchvision 
from facenet_pytorch import MTCNN, InceptionResnetV1
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import datasets

In [23]:
print("Platform:", sys.platform)
print("Python version:", sys.version)
print("---")
print("PIL version : ", PIL.__version__)
print("torch version : ", torch.__version__)
print("torchvision version : ", torchvision.__version__)

Platform: win32
Python version: 3.12.1 (tags/v3.12.1:2305ca5, Dec  7 2023, 22:03:25) [MSC v.1937 64 bit (AMD64)]
---
PIL version :  10.2.0
torch version :  2.2.2+cpu
torchvision version :  0.17.2+cpu


In [24]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using {device} device.")

Using cpu device.


**Initializing MTCNN and Inception-ResNet V1**

In [25]:
mtcnn0 = MTCNN(image_size=240,device=device, keep_all=False, min_face_size=40, post_process=False)
print(mtcnn0)

MTCNN(
  (pnet): PNet(
    (conv1): Conv2d(3, 10, kernel_size=(3, 3), stride=(1, 1))
    (prelu1): PReLU(num_parameters=10)
    (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=True)
    (conv2): Conv2d(10, 16, kernel_size=(3, 3), stride=(1, 1))
    (prelu2): PReLU(num_parameters=16)
    (conv3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1))
    (prelu3): PReLU(num_parameters=32)
    (conv4_1): Conv2d(32, 2, kernel_size=(1, 1), stride=(1, 1))
    (softmax4_1): Softmax(dim=1)
    (conv4_2): Conv2d(32, 4, kernel_size=(1, 1), stride=(1, 1))
  )
  (rnet): RNet(
    (conv1): Conv2d(3, 28, kernel_size=(3, 3), stride=(1, 1))
    (prelu1): PReLU(num_parameters=28)
    (pool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=True)
    (conv2): Conv2d(28, 48, kernel_size=(3, 3), stride=(1, 1))
    (prelu2): PReLU(num_parameters=48)
    (pool2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=True)
    (conv3): Conv2d(48, 64,

In [26]:
resnet = InceptionResnetV1(pretrained="vggface2").eval()

  0%|          | 0.00/107M [00:00<?, ?B/s]

In [28]:
print(resnet)

InceptionResnetV1(
  (conv2d_1a): BasicConv2d(
    (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
  )
  (conv2d_2a): BasicConv2d(
    (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
  )
  (conv2d_2b): BasicConv2d(
    (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
  )
  (maxpool_3a): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2d_3b): BasicConv2d(
    (conv): Conv2d(64, 80, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(80, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
  )
  (conv2d_4a): 

**Preparing Data**

In [30]:
images_folder = Path("data", "images")

print(f"Path to images: {images_folder}")

Path to images: data\images


ImageFolder object:

In [31]:
dataset = datasets.ImageFolder(images_folder)

print(dataset)

Dataset ImageFolder
    Number of datapoints: 10
    Root location: data\images


With the ImageFolder object, each subdirectory of the input path is considered a separate class. The class label is just the name of the subdirectory. 

Printing subdirectories:

In [32]:
for subdirectory in images_folder.iterdir():
    print(subdirectory)

data\images\mary_kom
data\images\ranveer


With a ImageFolder object, the .class_to_idx is a mapping between class label to class integer.

In [33]:
dataset.class_to_idx

{'mary_kom': 0, 'ranveer': 1}

However, we'd like to create the reverse mapping. In other words, integer to class label. Therefore, we create a dictionary that maps integer to class label.

In [34]:
idx_to_class = {i: c for c, i in dataset.class_to_idx.items()}

print(idx_to_class)

{0: 'mary_kom', 1: 'ranveer'}


We now create a DataLoader object with our ImageFolder. These objects are iterables that work well with PyTorch.

In [36]:
def collate_fn(x):
    return x[0]

In [37]:
loader = DataLoader(dataset, collate_fn=collate_fn)

print(loader.dataset)

Dataset ImageFolder
    Number of datapoints: 10
    Root location: data\images
